In [ ]:
# файл: test_generator_benchmarks.py

import json
import torch
from tqdm import tqdm
from datasets import load_dataset, get_dataset_config_names
from transformers import AutoModelForCausalLM, AutoTokenizer
from sklearn.metrics import accuracy_score, f1_score, mean_squared_error, r2_score

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Загрузка модели и токенизатора
model_name = "GenerTeam/GENERator-eukaryote-1.2b-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name).to(device)

def batch_predict_zero_shot(model, tokenizer, seqs, label_texts, prompt_template, batch_size):
    """Zero-shot классификация: для каждой последовательности генерируем продолжение и выбираем ближайший по косинусному сходству лейбл."""
    all_preds = []
    for i in tqdm(range(0, len(seqs), batch_size), desc="Predicting zero-shot"):
        batch_seqs = seqs[i:i+batch_size]
        prompts = [prompt_template.format(seq=seq, labels="; ".join(label_texts)) for seq in batch_seqs]
        enc = tokenizer(prompts, return_tensors="pt", padding=True).to(device)
        out = model.generate(**enc, max_new_tokens=64)
        texts = tokenizer.batch_decode(out, skip_special_tokens=True)
        # Берём текст после "Answer:" и выбираем ближайший лейбл
        for text in texts:
            answer = text.split("Answer:")[-1].strip().split()[0]
            # находим label_text с минимальным расстоянием Левенштейна или косинусным – здесь простая ==
            pred = answer if answer in label_texts else label_texts[0]
            all_preds.append(pred)
    return all_preds

def test_nt_tasks(model, tokenizer, batch_size=16, save_path="./results"):
    """Тестирование InstaDeepAI/nucleotide_transformer_downstream_tasks."""
    ds = "InstaDeepAI/nucleotide_transformer_downstream_tasks"
    configs = get_dataset_config_names(ds, trust_remote_code=True)
    results = {}
    prompt = "Sequence: {seq}\nLabels: {labels}\nAnswer:"
    for cfg in configs:
        data = (load_dataset(ds, trust_remote_code=True)
                if len(configs)==1 else load_dataset(ds, name=cfg, trust_remote_code=True))
        split = 'test' if 'test' in data else 'validation'
        seqs = data[split]['sequence']
        labels = data[split]['label']
        label_texts = [str(l) for l in sorted(set(labels))]
        preds = batch_predict_zero_shot(model, tokenizer, seqs, label_texts, prompt, batch_size)
        results[cfg] = {
            'accuracy': accuracy_score(labels, preds),
            'f1': f1_score(labels, preds, average='macro')
        }
    with open(f'{save_path}/nt_tasks_results.json', 'w') as f:
        json.dump(results, f, indent=2)
    return results

def test_lrb_tasks(model, tokenizer, batch_size=16, save_path="./results"):
    """Тестирование InstaDeepAI/genomics-long-range-benchmark (только классификационные задачи)."""
    ds = "InstaDeepAI/genomics-long-range-benchmark"
    # перечисляем только классификационные задачи
    class_tasks = [
        "variant_effect_causal_eqtl",
        "variant_effect_pathogenic_clinvar",
        "variant_effect_pathogenic_omim",
        "chromatin_features_histone_marks"
    ]
    results = {}
    prompt = "Sequence: {seq}\nLabels: {labels}\nAnswer:"
    for task in class_tasks:
        data = load_dataset(ds, task, trust_remote_code=True)
        split = 'test' if 'test' in data else 'validation'
        seqs = data[split]['sequence']
        labels = data[split]['label']
        label_texts = [str(l) for l in sorted(set(labels))]
        preds = batch_predict_zero_shot(model, tokenizer, seqs, label_texts, prompt, batch_size)
        results[task] = {
            'accuracy': accuracy_score(labels, preds),
            'f1': f1_score(labels, preds, average='macro')
        }
    with open(f'{save_path}/lrb_classification_results.json', 'w') as f:
        json.dump(results, f, indent=2)
    return results

if __name__ == "__main__":
    import os
    os.makedirs("./results", exist_ok=True)
    print("NT tasks:", test_nt_tasks(model, tokenizer, batch_size=16))
    print("LRB classification:", test_lrb_tasks(model, tokenizer, batch_size=16))


The repository for GenerTeam/GENERator-eukaryote-1.2b-base contains custom code which must be executed to correctly load the model. You can inspect the repository content at https://hf.co/GenerTeam/GENERator-eukaryote-1.2b-base.
You can avoid this prompt in future by passing the argument `trust_remote_code=True`.

Do you wish to run the custom code? [y/N]  y


README.md: 0.00B [00:00, ?B/s]

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

...

test.parquet:   0%|          | 0.00/838k [00:00<?, ?B/s]

test.parquet:   0%|          | 0.00/655k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/461850 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/48797 [00:00<?, ? examples/s]

Predicting zero-shot:   0%|          | 0/3050 [00:00<?, ?it/s]Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.
Predicting zero-shot:   0%|          | 1/3050 [00:03<3:13:07,  3.80s/it]Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.
Predicting zero-shot:   0%|          | 2/3050 [00:06<2:53:25,  3.41s/it]Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.
Predicting zero-shot:   0%|          | 3/3050 [00:10<2:46:56,  3.29s/it

In [ ]:
import numpy as np
import json
from datasets import load_dataset
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
from tqdm import tqdm
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

class GeneratorEmbeddingExtractor:
    """
    Извлекает эмбеддинги из модели GENERator.
    По умолчанию делает mean-pooling по всем токенам.
    """
    def __init__(self,
                 model_name: str = "GenerTeam/GENERator-eukaryote-1.2b-base",
                 device: torch.device = None,
                 pooling: str = "mean"):
        self.device = device or (torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu'))
        self.tokenizer = AutoTokenizer.from_pretrained(model_name, remote_trust_code=True)
        # Включаем output_hidden_states=False, т.к. нам нужен только последний слой
        self.model = AutoModelForCausalLM.from_pretrained(model_name, output_hidden_states=True, remote_trust_code=True).to(self.device)
        self.pooling = pooling

    def extract_embeddings(self, seqs, batch_size: int = 8):
        all_embs = []
        self.model.eval()
        with torch.no_grad():
            for i in tqdm(range(0, len(seqs), batch_size), desc="Extracting embeddings"):
                batch = seqs[i:i+batch_size]
                enc = self.tokenizer(
                    batch,
                    return_tensors="pt",
                    padding=True,
                    truncation=True
                ).to(self.device)

                out = self.model(**enc, return_dict=True)
                # последний скрытый слой [batch_size, seq_len, hidden_size]
                last_hidden = out.hidden_states[-1]

                if self.pooling == "mean":
                    mask = enc.attention_mask.unsqueeze(-1).float()  # (B, L, 1)
                    summed = (last_hidden * mask).sum(dim=1)
                    counts = mask.sum(dim=1).clamp(min=1)
                    emb = summed / counts
                else:
                    emb = last_hidden[:, 0, :]

                all_embs.append(emb.cpu().numpy())
        return np.vstack(all_embs)


ds = load_dataset("InstaDeepAI/nucleotide_transformer_downstream_tasks", trust_remote_code=True)
train_ds, test_ds = ds["train"], ds["test"]

extractor = GeneratorEmbeddingExtractor()
PARAMS_LOGREG = {"max_iter": 1000, "random_state": 42}
PATH_TO_SAVE_OUTPUTS = '/kaggle/working'
BATCH_SIZE = 16

# Baseline: обучение на полном наборе
baseline = {}
for task in tqdm(set(train_ds["task"]), desc="Baseline"):
    tr = train_ds.filter(lambda x, t=task: x["task"] == t)
    te = test_ds.filter(lambda x, t=task: x["task"] == t)
    seqs_tr, y_tr = tr["sequence"], np.array(tr["label"])
    seqs_te, y_te = te["sequence"], np.array(te["label"])

    X_tr = extractor.extract_embeddings(seqs_tr, batch_size=BATCH_SIZE)
    X_te = extractor.extract_embeddings(seqs_te, batch_size=BATCH_SIZE)

    clf = LogisticRegression(**PARAMS_LOGREG)

    Xf = X_tr.reshape(-1, 1) if X_tr.ndim == 1 or X_tr.shape[1] == 1 else X_tr
    Xt = X_te.reshape(-1, 1) if X_te.ndim == 1 or X_te.shape[1] == 1 else X_te
    clf.fit(Xf, y_tr)
    preds = clf.predict(Xt)

    baseline[task] = {
        "accuracy": float(accuracy_score(y_te, preds)),
        "f1_score": float(f1_score(y_te, preds, average="macro"))
    }
    with open(f"{PATH_TO_SAVE_OUTPUTS}/results_generator_task-{task}_baseline.json", "w") as f:
        json.dump(baseline, f, indent=4)

# Few-shot эксперименты
def few_shot(train, test, ks=(1, 5, 10, 20), trials=5):
    res = {}
    rng = np.random.RandomState(42)
    for task in tqdm(set(train["task"]), desc="Few-shot"):
        tr = train.filter(lambda x, t=task: x["task"] == t)
        te = test.filter(lambda x, t=task: x["task"] == t)
        seqs_tr, y_tr = tr["sequence"], np.array(tr["label"])
        seqs_te, y_te = te["sequence"], np.array(te["label"])
        X_te = extractor.extract_embeddings(seqs_te, batch_size=BATCH_SIZE)

        res[task] = {}
        for k in ks:
            accs, f1s = [], []
            for _ in range(trials):
                idxs = []
                for lbl in np.unique(y_tr):
                    locs = np.where(y_tr == lbl)[0]
                    choice = rng.choice(locs, size=min(k, len(locs)), replace=False)
                    idxs.extend(choice.tolist())

                X_k = extractor.extract_embeddings([seqs_tr[i] for i in idxs], batch_size=BATCH_SIZE)
                y_k = y_tr[idxs]

                clf = LogisticRegression(**PARAMS_LOGREG)
                Xf = X_k.reshape(-1, 1) if X_k.ndim == 1 or X_k.shape[1] == 1 else X_k
                Xt = X_te.reshape(-1, 1) if X_te.ndim == 1 or X_te.shape[1] == 1 else X_te
                clf.fit(Xf, y_k)
                p = clf.predict(Xt)
                accs.append(accuracy_score(y_te, p))
                f1s.append(f1_score(y_te, p, average="macro"))

            res[task][k] = {
                "accuracy": float(np.mean(accs)),
                "f1_score": float(np.mean(f1s))
            }
            with open(f"{PATH_TO_SAVE_OUTPUTS}/results_generator_task-{task}_k-{k}.json", "w") as f:
                json.dump(res, f, indent=4)

    return res

results_kshot = few_shot(train_ds, test_ds)

output = {"full": baseline, "kshot": results_kshot, "params": PARAMS_LOGREG}
with open(f"{PATH_TO_SAVE_OUTPUTS}/results_generator.json", "w") as f:
    json.dump(output, f, indent=4)


README.md: 0.00B [00:00, ?B/s]

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

train.parquet:   0%|          | 0.00/7.70M [00:00<?, ?B/s]

train.parquet:   0%|          | 0.00/6.72M [00:00<?, ?B/s]

...

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

Baseline:   0%|          | 0/18 [00:00<?, ?it/s]

Filter:   0%|          | 0/461850 [00:00<?, ? examples/s]

Filter:   0%|          | 0/48797 [00:00<?, ? examples/s]


Extracting embeddings:   0%|          | 0/1918 [00:00<?, ?it/s]Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.

Extracting embeddings:   2%|▏         | 29/1918 [00:14<15:24,  2.04it/s]